# E-Commerce Lakehouse: Silver Orders

## Inspect order statuses

Read the saved Bronze orders and inspect the distinct order statuses.
Understand the source values before applying cleaning rules.

In [0]:
bronze_orders_df = spark.table("workspace.ecommerce.bronze_orders")

### Inspect distinct order statuses

List the unique order_status values to understand how orders are
classified before defining cleaning rules.

In [0]:
bronze_orders_df.select("order_status").distinct()

DataFrame[order_status: string]

In [0]:
display(bronze_orders_df.select("order_status").distinct())

order_status
delivered
invoiced
shipped
processing
unavailable
canceled
created
approved


### Count orders by status

Count the orders in each status to understand their distribution
and establish a baseline for later validation.

In [0]:
bronze_orders_df.groupBy("order_status").count()

DataFrame[order_status: string, count: bigint]

In [0]:
display(bronze_orders_df.groupBy("order_status").count())

order_status,count
delivered,96478
invoiced,314
shipped,1107
processing,301
unavailable,609
canceled,625
created,5
approved,2


## Inspect purchase timestamp values

Preview the original purchase timestamp strings before choosing
a conversion format for the Silver table.

In [0]:
display(bronze_orders_df.select("order_purchase_timestamp").limit(5))

order_purchase_timestamp
2017-10-02 10:56:33
2018-07-24 20:41:37
2018-08-08 08:38:49
2017-11-18 19:28:06
2018-02-13 21:18:39


## Convert the purchase timestamp

Convert order_purchase_timestamp from a string to a timestamp
using the source format yyyy-MM-dd HH:mm:ss.
Keep the Bronze table unchanged.

In [0]:
from pyspark.sql.functions import to_timestamp

In [0]:
silver_orders_df = bronze_orders_df.withColumn("order_purchase_timestamp",
to_timestamp("order_purchase_timestamp", "yyyy-MM-dd HH:mm:ss")
)

### Verify the purchase timestamp type

Inspect the Silver DataFrame schema to confirm that
order_purchase_timestamp has changed from string to timestamp.

In [0]:
silver_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



### Check for missing purchase timestamps

Count rows where order_purchase_timestamp is null after conversion.
Investigate any missing values before continuing.

In [0]:
from pyspark.sql.functions import col

In [0]:
print(
    silver_orders_df.filter(
        col("order_purchase_timestamp").isNull()
    ).count()
)

0


### Inspect order approval values

Preview order_approved_at before converting it to a timestamp.
Missing approval dates may reflect the order's lifecycle and
should not automatically cause an order to be removed.

In [0]:
display(silver_orders_df.select("order_approved_at").limit(5))

order_approved_at
2017-10-02 11:07:15
2018-07-26 03:24:27
2018-08-08 08:55:23
2017-11-18 19:45:59
2018-02-13 22:20:29


### Convert the approval timestamp

Convert order_approved_at from string to timestamp using
yyyy-MM-dd HH:mm:ss. Preserve the purchase timestamp conversion.

In [0]:
silver_orders_df = silver_orders_df.withColumn("order_approved_at",
to_timestamp("order_approved_at", "yyyy-MM-dd HH:mm:ss")
)

### Verify the approval timestamp type

Inspect the schema to confirm that order_approved_at is now
a timestamp and the purchase timestamp remains correctly typed.

In [0]:
silver_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



### Check for missing approval timestamps

Count orders with a null approval timestamp.
Keep these records while investigating why approval dates are missing.

In [0]:
from pyspark.sql.functions import col
silver_orders_df.filter(col("order_approved_at").isNull()).count()

160

### Investigate missing approval timestamps by status

Group orders with null approval timestamps by order_status.
Use the results to understand the missing values before deciding
how to handle them.

In [0]:
display(
    silver_orders_df
    .filter(col("order_approved_at").isNull())
    .groupBy("order_status")
    .count()
)

order_status,count
canceled,141
delivered,14
created,5


### Check missing approval dates in the source

Count null order_approved_at values in the Bronze DataFrame
and compare the result with the 160 nulls found after conversion.

In [0]:
print(
bronze_orders_df
.filter(col("order_approved_at").isNull())
.count()
)

160


### Approval timestamp validation results

- Bronze and Silver each contain 160 null approval timestamps.
- The conversion introduced no additional null values.
- These include 141 canceled, 14 delivered, and 5 created orders.
- Preserve these records without filling in unknown approval dates.
- Missing approval dates on delivered orders remain a data quality concern.

### Inspect carrier handoff dates

Preview order_delivered_carrier_date to check its source format
before converting it from string to timestamp.

In [0]:
display(
silver_orders_df.select("order_delivered_carrier_date").limit(5)
)

order_delivered_carrier_date
2017-10-04 19:55:00
2018-07-26 14:31:00
2018-08-08 13:50:00
2017-11-22 13:39:59
2018-02-14 19:46:34


### Convert the carrier handoff timestamp

Convert order_delivered_carrier_date from string to timestamp.
Preserve the existing purchase and approval timestamp conversions.

In [0]:
silver_orders_df = silver_orders_df.withColumn("order_delivered_carrier_date",
to_timestamp("order_delivered_carrier_date","yyyy-MM-dd HH:mm:ss")
)

### Verify the carrier handoff timestamp type

Inspect the schema to confirm that order_delivered_carrier_date
is now a timestamp.

In [0]:
silver_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



### Validate missing carrier handoff timestamps

Compare null counts for order_delivered_carrier_date in Bronze
and Silver to check whether conversion introduced additional nulls.
Preserve missing dates rather than inventing handoff times.

In [0]:
print(bronze_orders_df.filter(col("order_delivered_carrier_date").isNull()).count())
print(silver_orders_df.filter(col("order_delivered_carrier_date").isNull()).count())




1783
1783


### Inspect customer delivery dates

Preview order_delivered_customer_date to check its source format
before converting it from string to timestamp.

In [0]:
display(silver_orders_df.select("order_delivered_customer_date").limit(5)
)

order_delivered_customer_date
2017-10-10 21:25:13
2018-08-07 15:27:45
2018-08-17 18:06:29
2017-12-02 00:28:42
2018-02-16 18:17:02


### Convert the customer delivery timestamp

Convert order_delivered_customer_date from string to timestamp.
Preserve the timestamp conversions already applied to Silver.

In [0]:
silver_orders_df = silver_orders_df.withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date", "yyyy-MM-dd HH:mm:ss")
)

### Validate the customer delivery timestamp

Confirm that order_delivered_customer_date is a timestamp.
Compare Bronze and Silver null counts to check whether conversion
introduced additional missing values.


In [0]:
silver_orders_df.printSchema()
print(bronze_orders_df.filter(col("order_delivered_customer_date").isNull()).count())
print(silver_orders_df.filter(col("order_delivered_customer_date").isNull()).count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

2965
2965


### Inspect estimated delivery dates

Preview order_estimated_delivery_date to identify its source
format before converting it to a timestamp.

In [0]:
display(
silver_orders_df.select("order_estimated_delivery_date").limit(5)
)

order_estimated_delivery_date
2017-10-18 00:00:00
2018-08-13 00:00:00
2018-09-04 00:00:00
2017-12-15 00:00:00
2018-02-26 00:00:00


### Convert the estimated delivery timestamp

Convert order_estimated_delivery_date from string to timestamp
using yyyy-MM-dd HH:mm:ss. Preserve the existing Silver conversions.

In [0]:
silver_orders_df = silver_orders_df.withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date", "yyyy-MM-dd HH:mm:ss")
)

### Validate the estimated delivery timestamp

Confirm that order_estimated_delivery_date is a timestamp.
Compare null counts in Bronze and Silver to check whether the
conversion introduced additional missing values.

In [0]:
silver_orders_df.printSchema()
print(bronze_orders_df.filter(col("order_estimated_delivery_date").isNull()).count())
print(silver_orders_df.filter(col("order_estimated_delivery_date").isNull()).count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

0
0


## Validate order ID uniqueness

Group records by order_id and identify IDs appearing more than once.
Investigate repeated IDs before deciding whether records should
be removed or corrected.

In [0]:
duplicate_order_ids_df = (
silver_orders_df
.groupBy("order_id")
.count()
.filter("count >1")
)
display(duplicate_order_ids_df)

order_id,count


### Check for missing order IDs

Count records with null or blank order_id values.
Every order must have a usable identifier.

In [0]:
from pyspark.sql.functions import trim

In [0]:
print(
    silver_orders_df.filter(
        (col("order_id").isNull()) | (trim(col("order_id")) == "")
    ).count()
)

0


### Check for missing customer IDs

Count records with null or blank customer_id values.
Customer identifiers are needed to connect orders to customer data.

In [0]:
print(
    silver_orders_df.filter(
        (col("customer_id").isNull()) | (trim(col("customer_id")) == "")
    ).count()
)

0


## Check delivery date consistency

Identify orders where customer delivery occurred before purchase.
Investigate these records before making corrections or exclusions.

In [0]:
invalid_delivery_dates_df = silver_orders_df.filter(
col("order_delivered_customer_date") < col("order_purchase_timestamp")
)
print(invalid_delivery_dates_df.count())

0


### Check approval date consistency

Count orders where approval occurred before purchase.
Missing approval timestamps are excluded from this comparison.

In [0]:
invalid_approval_dates_df = silver_orders_df.filter(
col("order_approved_at") < col("order_purchase_timestamp")
)
print(invalid_approval_dates_df.count())

0


### Check carrier and customer delivery sequence

Identify orders where customer delivery is recorded before
carrier handoff. Investigate inconsistencies without changing
the original timestamps.

In [0]:
invalid_shipping_sequence_df = silver_orders_df.filter(
col("order_delivered_customer_date") < col("order_delivered_carrier_date")
)
print(invalid_shipping_sequence_df.count())

23


### Inspect shipping sequence inconsistencies

Inspect the 23 orders where customer delivery precedes carrier
handoff. Preserve the source values while investigating the issue.

In [0]:
display(
invalid_shipping_sequence_df.select(
"order_id",
"order_status",
"order_delivered_carrier_date",
"order_delivered_customer_date"
).limit(5)
)

order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28T16:57:58.000Z,2017-07-25T19:32:56.000Z
383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07T17:22:41.000Z,2017-07-06T14:27:51.000Z
cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20T19:22:02.000Z,2017-07-19T14:13:28.000Z
dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01T18:23:30.000Z,2017-07-26T18:09:10.000Z
5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20T23:03:42.000Z,2017-07-20T18:52:41.000Z


### Flag inconsistent shipping sequences

Add shipping_sequence_issue to identify orders where customer
delivery precedes carrier handoff. Preserve all original timestamps.
Leave the flag null when either timestamp is missing.

In [0]:
silver_orders_df = silver_orders_df.withColumn("shipping_sequence_issue", col("order_delivered_customer_date") < col("order_delivered_carrier_date")
)

In [0]:
print(
    silver_orders_df
    .filter("shipping_sequence_issue = true")
    .count()
)

23


## Validate the Silver row count

Compare Bronze and Silver row counts to confirm that timestamp
conversions and the quality flag have not removed or added orders.

In [0]:
print("Bronze rows:", bronze_orders_df.count())
print("Silver rows:", silver_orders_df.count())

Bronze rows: 99441
Silver rows: 99441


## Check whether the Silver table exists

Check for an existing silver_orders table before saving the
validated data to avoid accidentally duplicating records.

In [0]:
print(spark.catalog.tableExists("workspace.ecommerce.silver_orders"))

True


## Refresh the validated Silver orders table

Replace the Silver orders table with the complete validated dataset.
Check the row count and order ID uniqueness before writing.
Verify the saved row count after the refresh.

In [0]:
table_name = "workspace.ecommerce.silver_orders"

checks = silver_orders_df.selectExpr(
    "count(*) AS row_count",
    "count(DISTINCT order_id) AS unique_order_ids",
    """
    count(CASE
        WHEN order_id IS NULL OR trim(order_id) = ''
        THEN 1
    END) AS missing_order_ids
    """
).first()

if (
    checks["row_count"] != 99441
    or checks["unique_order_ids"] != checks["row_count"]
    or checks["missing_order_ids"] != 0
):
    raise ValueError(
        "Silver order validation failed. Stopped before replacing the table."
    )

(
    silver_orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_count = spark.table(table_name).count()

if saved_count != checks["row_count"]:
    raise ValueError("Saved Silver count does not match the validated data.")

print("Silver orders refreshed:", saved_count)

Silver orders refreshed: 99441


## Verify the saved Silver table

Read workspace.ecommerce.silver_orders and confirm that it
contains 99,441 orders and 23 flagged shipping inconsistencies.

In [0]:
saved_silver_orders_df = spark.table("workspace.ecommerce.silver_orders")
print("Saved Silver rows:", saved_silver_orders_df.count())
print(
"Flagged shipping issues: ",
saved_silver_orders_df.filter("shipping_sequence_issue = true").count()
)

Saved Silver rows: 99441
Flagged shipping issues:  23
